# 01_data_prep.ipynb

## Setup

In [ ]:
import os
import sys
from pathlib import Path

# Mount colab
if True:
    if not os.path.exists('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')

PROJECT_ROOT: Path = Path("/content/drive/MyDrive/python/projects/nlp_projects/med/biomedical-extraction-pipeline")
os.chdir(PROJECT_ROOT)

# Add the PROJECT_ROOT directory to the Python path to import modules
if str(PROJECT_ROOT) in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"ℹ️ Current Python version: {sys.version}")

In [ ]:
# Install reqs (NOTE: restart after this)
if True:
    %pip install -r {"requirements/01_data_prep.txt"}

In [ ]:
# --- src ---
from src.config import load_configs
from src.utils import dynamic_module_reloader, seed_everything
from src.data_utils.preprocessing import (
    ensure_validation_split,
    flatten_bc5cdr,
    create_labeled_candidate_pairs
    )

# --- other ---
import pandas as pd
import torch
from dotenv import load_dotenv
from pprint import pprint
from datasets import load_dataset, DatasetDict, Dataset

MODULES_TO_RELOAD = [
    "src",
    "src.config",
    "src.utils",
    "src.data_utils",
    "src.data_utils.preprocessing"
]

dynamic_module_reloader(modules=MODULES_TO_RELOAD, verbose=1)
config = load_configs()
load_dotenv()
seed_everything(config.env.seed)
pprint(config.to_dict())

# Constants
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

In [ ]:
!nvidia-smi

## Data

### Download the dataset (bigbio/BC5CDR)

In [ ]:
# Load the bigbio BC5CDR dataset
dataset = load_dataset(config.data.dataset_name, trust_remote_code=True, token=os.environ["HF_TOKEN"])
ensure_validation_split(dataset)

In [ ]:
# Show feature schema and a single example
print(f"Dataset shape: {dataset.shape}")
print("\nFeature schema (train):\n", dataset["train"].features)
print("\nTrain keys:", list(dataset["train"][0].keys()))
if False:
    print("\nExample :")
    pprint(dataset["train"][0]["passages"])

### Inspect dataset structure

In [ ]:
# Expand a single record to see the nested structure
sample = dataset["train"][0]
passages_df = pd.json_normalize(sample['passages'])
print("Sample passages structure:")
print(passages_df.head().to_markdown())
print("\nColumns:", passages_df.columns.tolist())

# Flatten all passages from the entire train split
all_passages = []
for record in dataset["train"]:
    all_passages.extend(record['passages'])

passages_df_full = pd.json_normalize(all_passages)
print(f"\nFull passages DataFrame shape: {passages_df_full.shape}")
print("\nFirst few rows:")
print(passages_df_full.head().to_markdown())
print("\nData types:")
print(passages_df_full.dtypes)

In [ ]:
if True:
    # Check the structure of entities and relations
    display(pd.json_normalize(passages_df_full['entities']))
    display(pd.json_normalize(passages_df_full['relations']))


### Generate flattened dataset

In [ ]:
# Flatten the passages using the utility function across all the splits
flattened_dataset = DatasetDict({split: flatten_bc5cdr(dataset[split]) for split in dataset.keys()})
flattened_dataset

### Generate labeled relationship dataset

In [ ]:
# Create the labeled relation extraction dataset from the flattened train split
relations_dataset = flattened_dataset.copy()
for key in relations_dataset.keys():
    # Create the labeled relation extraction dataset from the flattened train split
    temp_df = relations_dataset[key].to_pandas()
    all_relation_candidates = []
    for _, row in temp_df.iterrows():
        all_relation_candidates.extend(create_labeled_candidate_pairs(row))

    relations_dataset[key] = Dataset.from_pandas(pd.DataFrame(all_relation_candidates))

relations_dataset = DatasetDict(relations_dataset)
relations_dataset

In [ ]:
# Check head of train split to make sure it looks good
print(
    relations_dataset["train"]
    .to_pandas()
    .head(10)
    .to_markdown()
)

In [ ]:
if True:
    # Assert that both tags are in each masked_text
    for key in relations_dataset.keys():
        subset = relations_dataset[key].to_pandas()
        assert not subset.empty, 'No candidates generated'
        missing_mask = (~subset['masked_text'].str.contains('<chemical>', case=False)) | (~subset['masked_text'].str.contains('<disease>', case=False))
        if missing_mask.any():
            missing = subset[missing_mask]
            raise AssertionError(f"Some rows are missing tags: {len(missing)} rows in split {key}. Examples: {missing.head(5)['document_id'].tolist()}")
        # All checks passed
        print(f'All masked_text entries contain both <chemical> and <disease> tags in split {key}.')

### Save processed data

In [ ]:
# Save the flattened dataset to disk
flattened_dataset.save_to_disk(config.data.flattened_dataset_path)
relations_dataset.save_to_disk(config.data.relations_dataset_path)

print(f"Saved flattened dataset to {config.data.flattened_dataset_path}")
print(f"Saved relation extraction dataset to {config.data.relations_dataset_path}")

## Requirements (full snapshot)

In [ ]:
if True:
    !pip freeze > snapshots/01_data_prep.txt